# Demo: QMLMultiClase — Clasificación multiclase (Iris)

Notebook demostrativo que crea la instancia QMLMultiClase, entrena con las tres clases del dataset Iris
y muestra métricas básicas. Ajusta `epochs` y `shots` para que el demo sea rápido.

In [ ]:
from quantum_information.metrics.distances import mba_distance
import numpy as np
import pennylane as qml

x_train = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
y_train = np.array([0, 1, 2, 1])
test = [0.5, 0.5]

m, qubits_dato = x_train.shape
qubits_qram = int(np.ceil(np.log2(m)))
qubits_label = int(np.ceil(np.log2(len(np.unique(y_train)))))
n_totales = qubits_qram + qubits_dato + qubits_label + 1
print("Número total de wires:", n_totales)

dev = qml.device("default.qubit", wires=n_totales)

@qml.qnode(dev)
def circuit():
    mba_distance(x_train, test, tipo="multiclase", labels=y_train)
    for i in range(qubits_dato):
        qml.ctrl(qml.RY, control=qubits_qram + i)(np.pi / qubits_dato, wires=n_totales - 1)
    return qml.probs(wires=range(qubits_qram + qubits_dato, n_totales - 1))

print("Probabilidades de ejemplo:", circuit())

In [ ]:
# Asegurar que el paquete 'src' está en sys.path cuando se ejecuta desde examples/qml/
import sys
from pathlib import Path
nb_path = Path.cwd()
repo_root = nb_path
# buscar carpeta 'src' hacia arriba
for _ in range(6):
    if (repo_root / 'src').exists():
        break
    repo_root = repo_root.parent
src_path = repo_root / 'src'
if src_path.exists():
    sys.path.insert(0, str(src_path.resolve()))
else:
    # fallback: añadir repo root
    sys.path.insert(0, str(repo_root.resolve()))
print('Using import path:', sys.path[0])

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

from quantum_information.metrics.models.qml_multiclase import QMLMultiClase

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

iris = load_iris()
X = iris.data
y = iris.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Samples:", X_scaled.shape)
print("Clases disponibles:", np.unique(y))

In [ ]:
# Crear instancia ligera para demo multidimensional
model = QMLMultiClase(
    codigo="gray",
    noise=0.0,
    backend=None,
    noise_model=None,
    result="probs",
    shots=512,
    lr=0.1,
    epochs=2,  # pocas epochs para demo
    optimize=False,
    use_weights=False,
)
print("Model created:", model)


In [ ]:
# Entrenar y evaluar sobre división train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Matriz de confusión:\n", confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred))

In [ ]:
# Explorar algunas predicciones con sus probabilidades
for idx in range(min(5, len(y_test))):
    print(f"Muestra {idx}")
    print("  Verdadera:", y_test[idx])
    print("  Predicha:", y_pred[idx])
    print("  Probabilidades:", np.round(y_proba[idx], 3))


Notas:
- Si tu entorno no tiene PennyLane o backend compatible, las celdas de fit/predict pueden fallar.
- Reduce `epochs` y `shots` para pruebas rápidas. Para entrenamientos reales incrementa ambos.
- Revisa `src/quantum_information/metrics/models/qml_multiclase.py` si cambias firmas o parámetros del modelo.